In [1]:
import numpy as np
import requests
from sentence_transformers import SentenceTransformer
import nltk
from nltk.corpus import wordnet as wn
from nltk.stem import WordNetLemmatizer
from better_profanity import profanity
import os

os.environ["PYENCHANT_LIBRARY_PATH"] = "/opt/homebrew/lib/libenchant-2.2.dylib"
import enchant

nltk.download("wordnet", download_dir="/Users/jymt/Projects/logocce/nltk_data")

# Load contextual model once
contextual_model = SentenceTransformer("all-MiniLM-L6-v2")


def build_filtered_vocab(vocab_size=50000):
    lemmatizer = WordNetLemmatizer()
    english_dict = enchant.Dict("en_US")
    profanity.load_censor_words()

    # Load most frequent words
    freq_url = "https://raw.githubusercontent.com/hermitdave/FrequencyWords/master/content/2016/en/en_50k.txt"
    frequency_text = requests.get(freq_url).text.lower().splitlines()

    frequency_set = set()
    for line in frequency_text[:vocab_size]:
        word = line.split()[0]
        if word.isalpha() and word.islower():
            frequency_set.add(word)

    # Collect common nouns from WordNet
    noun_set = set()
    for synset in wn.all_synsets("n"):  # 'n' = noun synsets
        for lemma in synset.lemma_names():
            word = lemma.lower().replace("_", "")  # Normalize
            if word.isalpha() and 3 <= len(word) <= 7 and word.islower():
                if lemmatizer.lemmatize(word, "n") == word:  # Ensure it's singular
                    noun_set.add(word)

    # Intersection of frequency list & noun set
    common_nouns = noun_set.intersection(frequency_set)

    # Explicitly remove proper nouns or uncommon surnames
    filtered_vocab = set(word for word in common_nouns if english_dict.check(word))

    # Filter out profane words using better-profanity
    filtered_vocab = {
        word for word in filtered_vocab if not profanity.contains_profanity(word)
    }

    return filtered_vocab


def precompute_contextual_embeddings(vocab):
    word_list = list(vocab)
    embeddings_tensor = contextual_model.encode(
        word_list, batch_size=64, convert_to_numpy=True, show_progress_bar=True
    )
    embeddings_dict = dict(zip(word_list, embeddings_tensor))
    return embeddings_dict


/opt/homebrew/Caskroom/miniforge/base/envs/glove-playground/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/jymt/Projects/logocce/nltk_data...


In [3]:
# Build filtered vocab
vocab = build_filtered_vocab()

# Precompute contextual embeddings
contextual_vecs = precompute_contextual_embeddings(vocab)

Batches: 100%|██████████| 128/128 [00:01<00:00, 86.57it/s] 


In [4]:
import random

class LogocceGame:
    def __init__(self, vocab, contextual_vecs):
        self.vocab = vocab
        self.word_list = list(vocab)
        self.emb_matrix = np.stack([contextual_vecs[word] for word in self.word_list])
        self.word_to_index = {word: idx for idx, word in enumerate(self.word_list)}
        self.target_word = random.choice(self.word_list)
        self.guess_history = []
        self.guessed_words = set()
        self.wrong_guesses = 0
        self.ai_improvement = 0.5
        self.revealed_indices = set()

    def semantic_distance_vec(self, word_vec, all_vecs):
        dot_products = np.dot(all_vecs, word_vec)
        norms = np.linalg.norm(all_vecs, axis=1) * np.linalg.norm(word_vec)
        cosine_similarities = dot_products / norms
        return 1 - cosine_similarities

    def ai_play_word(self, player_word):
        target_vec = self.emb_matrix[self.word_to_index[self.target_word]]
        player_vec = self.emb_matrix[self.word_to_index[player_word]]

        target_dist = 1 - np.dot(target_vec, player_vec) / (np.linalg.norm(target_vec) * np.linalg.norm(player_vec))
        desired_distance = target_dist * (1 - self.ai_improvement)

        dist_to_target = self.semantic_distance_vec(target_vec, self.emb_matrix)
        dist_to_player = self.semantic_distance_vec(player_vec, self.emb_matrix)

        valid_mask = (dist_to_target < target_dist) & (dist_to_player < target_dist)
        guessed_indices = [self.word_to_index[word] for word in self.guessed_words.union({player_word, self.target_word})]
        valid_mask[guessed_indices] = False

        for idx, word in enumerate(self.word_list):
            if word in self.target_word:
                valid_mask[idx] = False

        if not valid_mask.any():
            return None

        diffs = np.abs(dist_to_target - desired_distance)
        diffs[~valid_mask] = np.inf

        best_idx = np.argmin(diffs)
        best_word = self.word_list[best_idx]

        self.ai_improvement = min(self.ai_improvement + 0.1, 0.9)

        return best_word

    def reveal_hint(self):
        while len(self.revealed_indices) < min(self.wrong_guesses - 1, len(self.target_word)):
            unrevealed = set(range(len(self.target_word))) - self.revealed_indices
            self.revealed_indices.add(random.choice(list(unrevealed)))
        hint = ' '.join(self.target_word[i] if i in self.revealed_indices else '_' for i in range(len(self.target_word)))
        return hint

    def player_guess(self, guess_word):
        if guess_word in self.guessed_words:
            return f"'{guess_word}' has already been guessed. Try another word."

        if guess_word not in self.vocab:
            return f"'{guess_word}' is not in the valid word list. Try another word."

        self.guessed_words.add(guess_word)
        player_vec = self.emb_matrix[self.word_to_index[guess_word]]
        target_vec = self.emb_matrix[self.word_to_index[self.target_word]]
        distance = self.semantic_distance_vec(player_vec, np.array([target_vec]))[0]

        self.guess_history.append({'player': 'human', 'word': guess_word, 'distance': round(distance, 3)})

        if distance < 0.1:
            return f"Correct! The word was '{self.target_word}'."

        self.wrong_guesses += 1

        ai_word = self.ai_play_word(guess_word)
        if ai_word is None:
            return "AI has no valid words left to guess. Game over."

        ai_vec = self.emb_matrix[self.word_to_index[ai_word]]
        ai_distance = self.semantic_distance_vec(ai_vec, np.array([target_vec]))[0]

        self.guessed_words.add(ai_word)
        self.guess_history.append({'player': 'AI', 'word': ai_word, 'distance': round(ai_distance, 3)})

        response = {
            'player_guess': guess_word,
            'player_distance': round(distance, 3),
            'ai_guess': ai_word,
            'ai_distance': round(ai_distance, 3),
            'closer': 'player' if distance < ai_distance else 'ai',
            'guess_history': self.guess_history.copy()
        }

        if self.wrong_guesses > 1:
            response['hint'] = self.reveal_hint()

        return response

In [5]:
game = LogocceGame(vocab, contextual_vecs)

In [10]:
game.player_guess('collard')

"Correct! The word was 'collard'."